# Модуль 7.5 — Мультимодальность: vision и audio

Домашка к [лекции 7.5](https://itrubnikov.github.io/Train_of_Thought/docs/modules/07-5-multimodal/). Вы соберёте **агента-бухгалтера**: он смотрит на PDF-чек (текстом или картинкой) и возвращает **структурированный JSON** с позициями и суммой — с валидацией и repair-повтором. Плюс увидите, что audio — это отдельный пайплайн (распознаём речь локально через Whisper, без ключа).

Все блоки рабочие — `Run all` проходит целиком. Картинки и чек **генерируются прямо в ноутбуке**, скачивать ничего не нужно.

**Что нужно:**
- **API-ключ MiniMax** (`MINIMAX_API_KEY`) для vision-вызовов. Без ключа vision-ячейки мягко пропускаются — `Run all` не ломается, а ASR и проверки работают у всех.
- Vision-вход (приём картинок через OpenAI-совместимый `image_url`) поддерживает `MiniMax-M3` — её и берём. У MiniMax есть и другие vision-модели (напр. `MiniMax-VL-01`); для домашки используем M3.
- Активность — в финальной секции **«Задачи — доработайте рабочий код»**.

In [ ]:
# Зависимости. На Colab/Kaggle matplotlib и pillow уже стоят, остальное ставим.
%pip install -q "openai>=1.40" python-dotenv pydantic pymupdf reportlab faster-whisper matplotlib pillow
print("[ok] зависимости установлены")

## Настройка ключа

Ключ берём из окружения, **не хардкодим**:
- **локально** — скопируйте `.env.example` в `.env` и впишите `MINIMAX_API_KEY`;
- **Colab** — `Secrets` (значок ключа слева), имя `MINIMAX_API_KEY`;
- **Kaggle** — `Add-ons → Secrets`, имя `MINIMAX_API_KEY`.

MiniMax OpenAI-совместим: тот же клиент `openai`, меняется только `base_url`.

In [ ]:
import os, io, json, base64, urllib.request

# .env (локально). В Colab/Kaggle ключ приходит из Secrets — см. ниже.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
except Exception:
    pass

# Colab Secrets
try:
    from google.colab import userdata  # type: ignore
    if not os.environ.get("MINIMAX_API_KEY"):
        os.environ["MINIMAX_API_KEY"] = userdata.get("MINIMAX_API_KEY") or ""
except Exception:
    pass

# Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient  # type: ignore
    if not os.environ.get("MINIMAX_API_KEY"):
        os.environ["MINIMAX_API_KEY"] = UserSecretsClient().get_secret("MINIMAX_API_KEY")
except Exception:
    pass

from openai import OpenAI

MODEL = "MiniMax-M3"                       # vision-модель MiniMax (приём картинок, OpenAI-совместимый image_url)
BASE_URL = "https://api.minimax.io/v1"     # OpenAI-совместимый эндпоинт
HAS_KEY = bool(os.environ.get("MINIMAX_API_KEY"))

client = OpenAI(api_key=os.environ.get("MINIMAX_API_KEY", ""), base_url=BASE_URL) if HAS_KEY else None
print(f"[ok] окружение готово | MINIMAX_API_KEY: {'есть' if HAS_KEY else 'нет (vision-ячейки пропустятся)'}")

## Хелперы: картинка как content part

Главная идея модуля: картинка — это **не новый API**, а ещё один кусок (`content part`) в том же `messages`-вызове. Кладём в `content` список из `{type:text}` и `{type:image_url}`. Картинку упаковываем прямо в запрос как data-URI (MiniMax принимает base64 до 10 МБ).

`thinking: disabled` выключает reasoning у M3 — ответ приходит быстрее (ниже латентность), а без reasoning-токенов выходит меньше output, то есть дешевле по счёту (ставка за токен при этом та же). Нам нужны не рассуждения, а результат.

In [ ]:
def as_data_uri(png_bytes: bytes) -> str:
    """PNG-байты -> data-URI, чтобы отдать картинку прямо в запросе."""
    b64 = base64.b64encode(png_bytes).decode()
    return f"data:image/png;base64,{b64}"

def img_part(png_bytes: bytes, detail: str = "high") -> dict:
    return {"type": "image_url", "image_url": {"url": as_data_uri(png_bytes), "detail": detail}}

def text_part(s: str) -> dict:
    return {"type": "text", "text": s}

def ask_vision(parts, max_tokens: int = 512):
    """Один мультимодальный вызов. Возвращает (текст, usage) или (None, None) без ключа."""
    if not HAS_KEY:
        print("[skip] нет MINIMAX_API_KEY — пропускаю vision-вызов")
        return None, None
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": parts}],
        max_tokens=max_tokens,
        extra_body={"thinking": {"type": "disabled"}},  # M3: ответ без reasoning
    )
    return resp.choices[0].message.content, resp.usage

print("[ok] хелперы готовы: as_data_uri, img_part, text_part, ask_vision")

## Часть 1. Картинка как content part

Нарисуем график (matplotlib) и попросим модель описать тренд. Это самый маленький мультимодальный вызов: текстовый вопрос + картинка в одном `content`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

months = ["янв", "фев", "мар", "апр", "май", "июн"]
sales = [120, 135, 128, 160, 205, 280]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(months, sales, marker="o")
ax.set_title("Выручка по месяцам, тыс. руб")
ax.grid(True, alpha=0.3)
cbuf = io.BytesIO(); fig.savefig(cbuf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig)
chart_png = cbuf.getvalue()
print(f"[ok] график нарисован: {len(chart_png)} байт PNG")
display(IPyImage(data=chart_png))

In [ ]:
answer, usage = ask_vision(
    [text_part("Что показывает график? Назови тренд одним предложением."), img_part(chart_png)]
)
if answer is not None:
    print("Ответ модели:\n", answer)
    print("\nusage:", usage)   # у MiniMax поля prompt_tokens / completion_tokens

## Часть 2. OCR глазами модели

Классический OCR ищет буквы по форме и ломается на кривой вёрстке. Vision-модель «читает» картинку целиком. Сделаем картинку с текстом (PIL) и попросим переписать его дословно.

In [ ]:
from PIL import Image, ImageDraw

img = Image.new("RGB", (420, 140), "white")
d = ImageDraw.Draw(img)
d.text((14, 20), "INVOICE #2026-0042", fill="black")
d.text((14, 55), "Service: consulting", fill="black")
d.text((14, 90), "Total: 500 RUB", fill="black")
ibuf = io.BytesIO(); img.save(ibuf, "PNG"); ocr_png = ibuf.getvalue()
print(f"[ok] картинка с текстом готова: {len(ocr_png)} байт PNG")
display(IPyImage(data=ocr_png))

In [ ]:
answer, _ = ask_vision(
    [text_part("Перепиши весь текст с картинки дословно, сохрани строки."), img_part(ocr_png)]
)
if answer is not None:
    print("Распознанный текст:\n", answer)

## Часть 3. PDF — картинка или текст?

Главный практический выбор. PDF бывает двух видов:
- **текстовый** (сгенерирован программой) — текст достаётся мгновенно и бесплатно парсером;
- **скан** (фото страницы) — букв внутри нет, нужно рендерить в картинку и смотреть глазами модели.

Сгенерируем текстовый PDF-чек (reportlab) и пройдём оба пути одним инструментом — `pymupdf`.

In [ ]:
from reportlab.lib.pagesizes import A6
from reportlab.pdfgen import canvas

def make_receipt_pdf() -> bytes:
    buf = io.BytesIO()
    c = canvas.Canvas(buf, pagesize=A6)
    w, h = A6
    c.setFont("Helvetica-Bold", 12); c.drawString(20, h - 30, "CAFE TRAIN OF THOUGHT")
    c.setFont("Helvetica", 10)
    rows = [("Coffee", 1, 200.0), ("Croissant", 2, 150.0)]
    y = h - 60
    for name, qty, price in rows:
        # печатаем явно: количество x цена за единицу = сумма строки
        c.drawString(20, y, f"{name}  {qty} x {price:.2f}")
        c.drawRightString(w - 20, y, f"{qty * price:.2f}")
        y -= 18
    c.setFont("Helvetica-Bold", 11)
    c.drawString(20, y - 6, "TOTAL"); c.drawRightString(w - 20, y - 6, "500.00")
    c.showPage(); c.save()
    return buf.getvalue()

pdf_bytes = make_receipt_pdf()
print(f"[ok] PDF-чек сгенерирован: {len(pdf_bytes)} байт")

In [ ]:
import pymupdf  # fitz — старое имя того же пакета (import fitz ещё работает, но устарел)

doc = pymupdf.open(stream=pdf_bytes, filetype="pdf")
page = doc[0]

# Путь 1 — ТЕКСТ: мгновенно и бесплатно, если PDF «текстовый»
pdf_text = page.get_text()
has_text = bool(pdf_text.strip())

# Путь 2 — КАРТИНКА: рендерим страницу в PNG (для сканов)
pdf_png = page.get_pixmap(dpi=150).tobytes("png")

print(f"[path text]  get_text() непустой: {has_text}")
print("текст из PDF:\n" + pdf_text)
print(f"[path image] рендер страницы: {len(pdf_png)} байт PNG")
display(IPyImage(data=pdf_png))

## Часть 4. Чек → JSON (артефакт)

Агенту нужна **структура**, а не простыня. Описываем форму схемой (`pydantic`), просим JSON, на выходе **валидируем** — при сбое один repair-повтор (как structured output в 6.5, только источник теперь может быть картинкой).

Функция от источника не зависит: ей всё равно, пришёл текст PDF или картинка-рендер — она про **форму ответа**.

In [ ]:
import re
from pydantic import BaseModel, ValidationError

class Item(BaseModel):
    name: str
    qty: int
    price: float

class Receipt(BaseModel):
    items: list[Item]
    total: float

SCHEMA_HINT = '{"items":[{"name":str,"qty":int,"price":float}],"total":float}'

def extract_json_block(text: str) -> str:
    """Достаём чистый JSON: снимаем markdown-ограждение, <think>-префикс и прозу вокруг."""
    s = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    i, j = s.find("{"), s.rfind("}")
    if i != -1 and j != -1 and j > i:          # вырезаем от первой { до последней }
        s = s[i:j + 1]
    return s.strip()

print("[ok] схема Receipt / Item + extract_json_block готовы")

In [ ]:
def extract_receipt(content_parts, max_rounds: int = 2) -> Receipt:
    """Чек -> Receipt. build-twice: при невалидном JSON — один repair-повтор."""
    parts = list(content_parts)
    last_raw = ""
    for _ in range(max_rounds):
        answer, _usage = ask_vision(parts, max_tokens=1024)
        if answer is None:                       # нет ключа — наверх
            raise RuntimeError("нет MINIMAX_API_KEY")
        last_raw = extract_json_block(answer)          # снимаем ограждение/<think>/прозу
        try:
            return Receipt.model_validate_json(last_raw)  # Pydantic v2: кривой JSON тоже даёт ValidationError
        except ValidationError:
            parts = parts + [text_part(
                f"Это не прошло валидацию: {last_raw}. Верни ТОЛЬКО валидный JSON по схеме {SCHEMA_HINT}."
            )]
    raise ValueError(f"не удалось получить валидный JSON за {max_rounds} захода. Последний ответ: {last_raw}")

PROMPT = ("Перепиши чек в JSON строго по схеме " + SCHEMA_HINT +
          ". qty — целое количество; price — цена за ЕДИНИЦУ товара (не сумма по строке); "
          "total — итог по чеку. Только JSON, без пояснений.")

# json.dumps(model_dump()) — печатает UTF-8 на любом pydantic v2
# (model_dump_json(ensure_ascii=...) появился только в 2.12)
def show_json(r): print(json.dumps(r.model_dump(), indent=2, ensure_ascii=False))

# Текстовый путь (дёшево): отдаём извлечённый текст PDF
if HAS_KEY:
    receipt = extract_receipt([text_part(PROMPT + "\n\nЧек:\n" + pdf_text)])
    print("JSON из текстового PDF:")
    show_json(receipt)
else:
    print("[skip] нет ключа — пример JSON, который вернул бы агент:")
    receipt = Receipt(items=[Item(name="Coffee", qty=1, price=200.0),
                             Item(name="Croissant", qty=2, price=150.0)], total=500.0)
    show_json(receipt)

In [ ]:
def check_receipt(r: Receipt, tol: float = 0.01):
    """Инвариант: сумма позиций должна сходиться с total. Ловит галлюцинацию цифры."""
    calc = sum(i.qty * i.price for i in r.items)
    return abs(calc - r.total) <= tol, calc

# self-check — работает без ключа: проверяем саму проверку на заведомо верном чеке
_good = Receipt(items=[Item(name="A", qty=2, price=100.0)], total=200.0)
_bad  = Receipt(items=[Item(name="A", qty=2, price=100.0)], total=999.0)
assert check_receipt(_good)[0] is True
assert check_receipt(_bad)[0] is False
print("[ok] self-check инварианта пройден (без ключа)")

ok, calc = check_receipt(receipt)
print(f"проверка чека: сумма позиций = {calc}, total = {receipt.total}, сходится = {ok}")

## Часть 5. Audio: модель видит, но не слышит

Ни MiniMax-M3, ни Claude не принимают звук на вход. Поэтому audio — это **пайплайн**: `ASR (звук→текст) → LLM → (опц.) TTS`. Первое звено запускаем **локально и бесплатно** через `faster-whisper` — ключ не нужен.

Распознаём приложенную диктовку чека (`fixtures/voice.wav`). В Colab/Kaggle файла рядом нет — подтянем из репо.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress.*")  # косметика tqdm вне Jupyter-виджетов
from faster_whisper import WhisperModel

VOICE = "fixtures/voice.wav"
RAW = ("https://raw.githubusercontent.com/ITrubnikov/Train_of_Thought-homework/"
       "main/notebooks/module-7-5-multimodal/fixtures/voice.wav")
HAS_VOICE = os.path.exists(VOICE)
if not HAS_VOICE:                                  # Colab/Kaggle: качаем сэмпл из репо
    try:
        os.makedirs("fixtures", exist_ok=True)
        urllib.request.urlretrieve(RAW, VOICE)
        HAS_VOICE = True
        print("[ok] voice.wav скачан из репо")
    except Exception as e:                         # 404/оффлайн не валит Run all
        print(f"[skip] не удалось получить voice.wav ({e}) — ASR-ячейка пропущена")

voice_text = ""
if HAS_VOICE:
    asr = WhisperModel("base", device="cpu", compute_type="int8")  # модель скачается один раз
    segments, info = asr.transcribe(VOICE, language="ru")
    voice_text = " ".join(s.text for s in segments).strip()
    print(f"[ok] распознано (whisper base, lang={info.language}):")
    print(voice_text)

In [ ]:
# Дальше распознанный текст идёт в тот же агент, что и текст из PDF.
if not voice_text:
    print("[skip] нет распознанного текста (voice.wav недоступен) — пропускаю шаг агента")
elif HAS_KEY:
    voice_receipt = extract_receipt([text_part(PROMPT + "\n\nЧек (продиктован голосом):\n" + voice_text)])
    print("JSON из голосовой диктовки:")
    show_json(voice_receipt)
    ok, calc = check_receipt(voice_receipt)
    print(f"проверка: сумма = {calc}, total = {voice_receipt.total}, сходится = {ok}")
else:
    print("[skip] нет ключа — но видно главное: звук -> текст -> тот же LLM-вызов.")
    print("Распознанный текст, который ушёл бы в агент:\n", voice_text)

## Задачи — доработайте рабочий код

Весь код выше рабочий. Теперь меняйте его и записывайте, что изменилось (минимум 3 из 5):

1. **detail и токены.** В `img_part` поменяйте `detail="high"` на `"low"` для графика из Части 1. Сравните ответ и `usage` (prompt_tokens). Что потерялось, насколько упали токены?
2. **Текст vs картинка.** Прогоните `extract_receipt` на чеке двумя путями: текстовым (`pdf_text`) и картиночным (`[text_part(PROMPT), img_part(pdf_png)]`). Сравните JSON и стоимость. Когда какой путь выбрать?
3. **Новое поле схемы.** Добавьте в `Item` поле `category: str` (или в `Receipt` — `currency: str`). Обновите промпт и убедитесь, что валидация требует новое поле.
4. **Поймать галлюцинацию.** Сделайте «грязный» скан: возьмите `pdf_png`, через PIL размойте/зашумите цифры (`ImageFilter.GaussianBlur`), скормите картиночным путём и проверьте `check_receipt`. Поймали расхождение `total`?
5. **Своя диктовка.** Запишите свою голосовую заметку (телефон/диктофон), сохраните как `.wav`/`.mp3`, прогоните через `faster-whisper` и тот же агент.

## Что дальше

Вы научили модель смотреть: картинка как content part, PDF текстом или рендером, чек → валидированный JSON, audio как пайплайн ASR→LLM. Это те «глаза», которыми дальше пользуется агент.

→ [Модуль 8. Что такое агент](https://itrubnikov.github.io/Train_of_Thought/docs/modules/08-what-is-agent/) — модель начинает не только смотреть, но и **действовать**.

Критерии приёма — в `<HomeworkBlock>` лекции: ноутбук прогнан целиком, зафиксирован реальный JSON чека, сделаны задачи (≥3 из 5), ключ не захардкожен.